# SSAE Symbolic Logic Training

Trains the sparse autoencoder on symbolic logic reasoning traces.

**Runtime:** T4 (~45 min total) or A100 (~15 min total)  
**Output:** `ssae_symbolic_p1.enc.pt` (~3 MB) — download this file back to your local machine.

### Steps
1. Config & setup
2. Clone repo + install deps
3. Generate synthetic ProntoQA problems + model traces
4. Train SSAE Phase 1
5. Download the inference checkpoint

## 0 — Config
Set your GitHub repo URL and training parameters here.

In [1]:
# ── Edit these ────────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/djaxchi/CoT-checker.git" 

# Data generation
N_PROBLEMS    = 14000   # ~50K steps at avg 3.5 steps/problem
GEN_BATCH     = 16       # LLM generation batch size

# Training
TRAIN_EPOCHS  = 1
TRAIN_BATCH   = 16       # per-device batch size
GRAD_ACCUM    = 4        # effective batch = 16 * 4 = 64
# ── End config ────────────────────────────────────────────────────────────────

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU : {gpu}")
print(f"VRAM: {vram:.0f} GB")

# A100 (40 GB) can unfreeze backbones; T4 (16 GB) must keep them frozen
FREEZE_BACKBONES = vram < 35
print(f"Freeze backbones: {FREEZE_BACKBONES}  ({'T4/V100 — memory constrained' if FREEZE_BACKBONES else 'A100 — full training'})")

GPU : Tesla T4
VRAM: 16 GB
Freeze backbones: True  (T4/V100 — memory constrained)


## 1 — Clone repo + install dependencies

In [2]:
import os

!git clone {GITHUB_REPO} CoT-checker
%cd CoT-checker

!pip install -q transformers datasets tqdm

Cloning into 'CoT-checker'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 55 (delta 11), reused 53 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 1.00 MiB | 4.08 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/CoT-checker


## 2 — Generate synthetic ProntoQA problems + model traces

Phase A: generate problems algorithmically (no LLM needed, instant).  
Phase B: run Qwen2.5-0.5B on each problem to get model-generated CoT traces.

In [3]:
!python scripts/generate_symbolic_data.py \
    --n-problems {N_PROBLEMS} \
    --problems-out data/prontoqa_synthetic.jsonl \
    --traces-out   data/prontoqa_traces.jsonl \
    --batch-size {GEN_BATCH} \
    --device cuda

python3: can't open file '/content/CoT-checker/scripts/generate_symbolic_data.py': [Errno 2] No such file or directory


In [ ]:
# Sanity check: inspect a few traces
import json

with open("data/prontoqa_traces.jsonl") as f:
    samples = [json.loads(l) for l in f][:3]

total_steps = sum(len(s["steps"]) for s in (json.loads(l) for l in open("data/prontoqa_traces.jsonl")))
print(f"Total traces : {sum(1 for _ in open('data/prontoqa_traces.jsonl'))}")
print(f"Total steps  : {total_steps}")
print()
for s in samples:
    print(f"Q: {s['question']}")
    print(f"Steps: {s['steps']}")
    print()

## 3 — Train SSAE Phase 1

Trains the sparse autoencoder to reconstruct step tokens from a sparse latent `h_c`.  
The Qwen2.5-0.5B backbone is frozen on T4; unfrozen on A100.

In [ ]:
freeze_flag = "--freeze-backbones" if FREEZE_BACKBONES else "--no-freeze-backbones"

!python scripts/train_ssae_symbolic.py \
    --traces data/prontoqa_traces.jsonl \
    --output results/checkpoints/ssae_symbolic_p1.pt \
    --phase 1 \
    --epochs {TRAIN_EPOCHS} \
    --batch-size {TRAIN_BATCH} \
    --grad-accum {GRAD_ACCUM} \
    --device cuda \
    {freeze_flag}

In [ ]:
# Verify checkpoint files
import os
for f in os.listdir("results/checkpoints"):
    path = f"results/checkpoints/{f}"
    size = os.path.getsize(path) / 1e6
    print(f"{f:<50}  {size:6.1f} MB")

## 4 — Download the inference checkpoint

The `.enc.pt` file (~3 MB) is all you need locally to run the probe pipeline.  
Place it at `results/checkpoints/ssae_symbolic_p1.enc.pt` on your machine.

In [ ]:
from google.colab import files

# Download the inference checkpoint (~3 MB)
files.download("results/checkpoints/ssae_symbolic_p1.enc.pt")

# Optional: also download the traces for local probe-data generation
# files.download("data/prontoqa_traces.jsonl")

## 5 — (Optional) Generate probe data on Colab and download

If you want to skip re-running generation locally, encode the traces here and download the `.npz` (~few MB).

In [ ]:
# This uses the full checkpoint (not the slim .enc.pt) so it can re-use the
# backbone that's already in memory on Colab.

!python scripts/generate_probe_data_symbolic.py \
    --checkpoint results/checkpoints/ssae_symbolic_p1.pt \
    --data       data/prontoqa_synthetic.jsonl \
    --output     results/probe_data/symbolic_ssae_p1.npz \
    --device cuda

files.download("results/probe_data/symbolic_ssae_p1.npz")